# SFS, Fst and PBS

The data is from the 1000 Genomes Project, using three populations:

| population | description |
| --- | --- |
| **CEU** | Europeans (mostly of British ancestry) |
| **JPT** | East Asian - Japanese individuals |
| **YRI** | West African - Nigerian Yoruba individuals |

To keep the run times short we use a very reduced data set:

 - input data: **bam files** (aligned reads, not called genotypes)
 - **10 individuals** from each population
 - a reduced genome: 30 x 100 kb random regions across the autosomes, **plus one non-random region**
 - each individual sequenced at **2-6X**

### Aims

 1. Reconstruct the site frequency spectrum, in 1 and 2 dimensions.
 2. Estimate **Fst** between each pair of populations.
 3. Run a scan statistic, **PBS**, to detect signs of positive selection.

Everything is estimated from **genotype likelihoods**, so we never call genotypes or
variable sites - which matters at 2-6X, where a single read cannot tell a homozygote from
a heterozygote. The bonus at the end compares this with what you get if you *do* call
genotypes.

## 1. Setup

The first cell only defines paths. It writes them to a small file (`env.sh`) that every
later bash cell reads back with `source`, so nothing breaks if you restart the kernel or
run cells out of order. The file also does `cd` into your working folder.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA_PATH=/course/data/current_data/sfs_fst_pbs

# where you will do the exercise
WORK=~/sfs_fst_pbs_human
mkdir -p $WORK

cat > $WORK/env.sh <<EOF
# ---- data ----
export DATA_PATH=$DATA_PATH
export WORK=$WORK

# ---- programs (both are on PATH on this server) ----
export ANGSD=angsd
export REAL=realSFS

# ---- reference and ancestral fasta ----
export ANC=\$DATA_PATH/hg19ancNoChr.fa.gz     # ancestral states, from chimp
export REF=\$DATA_PATH/hg19.fa.gz             # human reference, hg19

# ---- the bam folders ----
export BAMFOLDER=\$DATA_PATH/smallerbams      # the reduced genome, all autosomes
export BAMFOLDERchr5=\$DATA_PATH/chr5_33M_v2  # the 1 Mb region on chromosome 5

# ---- precomputed results, in case the angsd runs take too long ----
export PRECOMP=\$DATA_PATH/precomputed

# ---- the R helper that draws the 2D spectra ----
export PLOT2DSFS=/course/data/current_data/scripts/plot2dSFS.R

# always work from the same folder
cd \$WORK
EOF

echo "--- wrote $WORK/env.sh ---"
cat $WORK/env.sh

Check that the programs and the data files are where we expect them. **Make sure there
are no error messages below** - if something is missing, everything after it will fail.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

echo "--- programs ---"
type angsd realSFS

echo -e "\n--- reference and ancestral fasta ---"
ls -l $ANC $REF

echo -e "\n--- bam folders ---"
echo -n "autosomes: "; ls $BAMFOLDER/*.bam | wc -l
echo -n "chr5:      "; ls $BAMFOLDERchr5/*.bam | wc -l

echo -e "\n--- working folder ---"
pwd

### Make the bam file lists

Most ANGSD analyses take a *bamlist*: a plain text file with one bam path per line. We
make one per population by grepping the population name out of the file names.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

# an African population
find $BAMFOLDER | grep bam$ | grep YRI > YRI.filelist
# an Asian population
find $BAMFOLDER | grep bam$ | grep JPT > JPT.filelist
# a European population
find $BAMFOLDER | grep bam$ | grep CEU > CEU.filelist

wc -l YRI.filelist JPT.filelist CEU.filelist
head -n 2 YRI.filelist

## 2. Reconstructing the site frequency spectrum

First set some filters to remove the worst reads (`-minMapQ`) and the worst bases
(`-minQ`).

Then set the options that say: calculate genotype likelihoods with the **GATK** model
(`-gl 2`) and from those calculate the **site allele frequency likelihoods** (`-dosaf 1`).
The saf files are the input to everything else in this exercise.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

FILTERS="-minMapQ 30 -minQ 20"
OPT=" -dosaf 1 -gl 2"

# keep them in env.sh too, so later cells can reuse them
echo "export FILTERS=\"$FILTERS\"" >> $WORK/env.sh
echo "export OPT=\"$OPT\""         >> $WORK/env.sh

echo "FILTERS: $FILTERS"
echo "OPT:    $OPT"

Generate the site allele frequency likelihoods with ANGSD, one run per population.
This takes a couple of minutes - the three runs are put in the background so they run at
the same time.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh
# ~2-3 min for all three

$ANGSD -b YRI.filelist -anc $ANC -out yri $FILTERS $OPT -ref $REF &
$ANGSD -b JPT.filelist -anc $ANC -out jpt $FILTERS $OPT -ref $REF &
$ANGSD -b CEU.filelist -anc $ANC -out ceu $FILTERS $OPT -ref $REF
wait

echo "--- created files ---"
ls -l *.saf.idx

If that takes too long you can copy the precomputed results instead, and carry on.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

## uncomment to use the precomputed saf files
# cp $PRECOMP/yri.saf* .
# cp $PRECOMP/jpt.saf* .
# cp $PRECOMP/ceu.saf* .
# ls -l *.saf.idx

echo "precomputed files available:"
ls $PRECOMP/*.saf.idx

### The 1D spectrum

Now estimate the site frequency spectrum for each population **directly from the site
allele frequency likelihoods** - without calling genotypes or variable sites.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

# the 1-dimensional SFS
$REAL yri.saf.idx > yri.sfs
$REAL jpt.saf.idx > jpt.sfs
$REAL ceu.saf.idx > ceu.sfs

echo "--- yri.sfs (20 individuals -> 21 categories, 0 to 20 derived alleles) ---"
cat yri.sfs

Plot the three spectra.

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

nnorm <- function(x) x/sum(x)

# expected number of sites with 1:20 derived alleles
res <- rbind(
  YRI = scan("yri.sfs")[-1],
  JPT = scan("jpt.sfs")[-1],
  CEU = scan("ceu.sfs")[-1]
)
colnames(res) <- 1:20

# density instead of expected counts
res <- t(apply(res, 1, nnorm))

# the non-ancestral sites
barplot(res, beside=T, legend=c("YRI","JPT","CEU"), names=1:20,
        main="realSFS non ancestral sites")

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

# the polymorphic sites only (drop the fixed-derived category)
resPoly <- t(apply(res[,-20], 1, nnorm))
barplot(resPoly, beside=T, legend=c("YRI","JPT","CEU"), names=1:19,
        main="realSFS polymorphic sites")

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

# because there is so little data, downsample to 5 individuals (10 chromosomes)
# and exclude the fixed derived category
downsampleSFS <- function(x, chr){   # x is 1:2n, chr < 2n
    n <- length(x)
    mat <- sapply(1:chr, function(i) choose(1:n,i)*choose(n-(1:n),chr-i)/choose(n,chr))
    nnorm( as.vector(t(mat) %*% x)[-chr] )
}
resDown <- t(apply(res, 1, downsampleSFS, chr=10))
barplot(resDown, beside=T, legend=c("YRI","JPT","CEU"), names=1:9,
        main="realSFS downsampled polymorphic sites")

**Questions**

 - Which population has the largest population size?
 - The data is a small subset of the genome (2 Mb). With 6 Mb it would have looked like
   [this](https://www.popgen.dk/albrecht/phdcourse/html/plots/realSFS4.pdf).
 - The same analysis on a whole chromosome for the 1000 Genomes individuals looks like
   [this](https://www.popgen.dk/albrecht/phdcourse/sfs/Moltke5V2.pdf).

### Statistics from the spectrum

The spectrum is a summary of the data, and several classical statistics are functions of
it. Here we get Watterson's theta and from it an effective population size.

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

## read the three spectra
y <- scan("yri.sfs")
j <- scan("jpt.sfs")
c <- scan("ceu.sfs")

x <- y   # change this to j or c to try the other populations

nSites <- sum(x)                 # number of sites where we have data
nSeg   <- sum(x[c(-1,-21)])      # number of segregating sites
an     <- function(n) sum(1/1:(n-1))
thetaW <- nSeg/an(20)            # Watterson's theta

cat("sites with data:   ", round(nSites), "\n")
cat("segregating sites: ", round(nSeg), "\n")
cat("fraction segregating:", signif(nSeg/nSites, 3), "\n")
cat("Watterson's theta: ", signif(thetaW, 4), "\n")
cat("effective population size:", round(thetaW / 1.5e-8 / nSites / 4), "\n")

The cell above is for the African population. Run it for all three by changing the `x <- y`
line.

**Questions**

 - Which has the largest population size?
 - Which has the largest variability, i.e. the largest fraction of polymorphic
   (segregating) sites?

## 3. Fst and PBS

To estimate Fst between two populations we first need the **2-dimensional** frequency
spectrum, again estimated from the site allele frequency likelihoods.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

# the 2D SFS for each pair
$REAL yri.saf.idx ceu.saf.idx > yri.ceu.ml &
$REAL yri.saf.idx jpt.saf.idx > yri.jpt.ml &
$REAL jpt.saf.idx ceu.saf.idx > jpt.ceu.ml
wait

echo "--- each file is a 21x21 matrix, so 441 numbers ---"
wc -w yri.ceu.ml yri.jpt.ml jpt.ceu.ml

Plot the three 2D spectra. The colours are densities: red and black mean many sites look
like that, green means few do.

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

yc <- scan("yri.ceu.ml")
yj <- scan("yri.jpt.ml")
jc <- scan("jpt.ceu.ml")

# the helper that draws the 2D spectrum
source("/course/data/current_data/scripts/plot2dSFS.R")

plot2 <- function(s, ...){
    dim(s) <- c(21,21)
    s[1] <- NA           # both populations ancestral
    s[21,21] <- NA       # both populations fixed derived
    s <- s/sum(s, na.rm=T)
    pal <- color.palette(c("darkgreen","#00A600FF","yellow","#E9BD3AFF",
                           "orange","red4","darkred","black"), space="rgb")
    pplot(s/sum(s, na.rm=T), pal=pal, ...)
}

plot2(yc, ylab="YRI", xlab="CEU")

In [ ]:
plot2(yj, ylab="YRI", xlab="JPT")

In [ ]:
plot2(jc, ylab="JPT", xlab="CEU")

Because there is so little data the plots are noisy, but they are still informative.

**Questions**, from the plots alone:

 - Which population has the most private SNPs, i.e. sites that are only polymorphic in
   that population?
 - Which two populations are the most closely related?

### Pairwise Fst

Now put a number on it. First index the pairs so that the same sites are used for each
population, then get the global estimate.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

# index the pairs, so the same sites are analysed for both populations
$REAL fst index jpt.saf.idx ceu.saf.idx -sfs jpt.ceu.ml -fstout jpt.ceu
$REAL fst index yri.saf.idx ceu.saf.idx -sfs yri.ceu.ml -fstout yri.ceu
$REAL fst index yri.saf.idx jpt.saf.idx -sfs yri.jpt.ml -fstout yri.jpt

echo -e "\n--- global Fst estimates (unweighted, weighted) ---"
echo -n "JPT vs CEU: "; $REAL fst stats jpt.ceu.fst.idx
echo -n "YRI vs JPT: "; $REAL fst stats yri.jpt.fst.idx
echo -n "YRI vs CEU: "; $REAL fst stats yri.ceu.fst.idx

Look at the **weighted** Fst (`Fst.Weight`), the second number.

**Questions**

 - Which two populations are the most closely related?
 - Which two are the most distantly related?

### Sliding windows across the genome

Fst and PBS vary along the genome. Let us look at that variation in 50 kb windows, so
that later we have a **background distribution** to compare a candidate region against.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh

# all three populations at once, so we can get PBS as well as pairwise Fst
$REAL fst index yri.saf.idx jpt.saf.idx ceu.saf.idx -fstout yri.jpt.ceu \
    -sfs yri.jpt.ml -sfs yri.ceu.ml -sfs jpt.ceu.ml

$REAL fst stats2 yri.jpt.ceu.fst.idx -win 50000 -step 10000 > slidingwindowBackground

echo "--- windows: $(wc -l < slidingwindowBackground) ---"
head -n 3 slidingwindowBackground

Read the windows into R and look at the distributions.

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

r <- read.delim("slidingwindowBackground", as.is=T, head=T)
names(r)[-c(1:4)] <- c("wFst_YRI_JPT","wFst_YRI_CEU","wFst_JPT_CEU",
                       "PBS_YRI","PBS_JPT","PBS_CEU")
head(r)

par(mfcol=c(3,2))

# the distribution of Fst
mmax <- max(c(r$wFst_YRI_JPT, r$wFst_YRI_CEU, r$wFst_JPT_CEU), na.rm=T)
hist(r$wFst_YRI_JPT, col="lavender",  xlim=c(0,mmax), br=20)
hist(r$wFst_YRI_CEU, col="mistyrose", xlim=c(0,mmax), br=20)
hist(r$wFst_JPT_CEU, col="hotpink",   xlim=c(0,mmax), br=20)

# the distribution of PBS
mmax <- max(c(r$PBS_CEU, r$PBS_YRI, r$PBS_JPT), na.rm=T)
hist(r$PBS_YRI, col="lavender",  xlim=c(0,mmax), br=20)
hist(r$PBS_CEU, col="mistyrose", xlim=c(0,mmax), br=20)
hist(r$PBS_JPT, col="hotpink",   xlim=c(0,mmax), br=20)

par(mfcol=c(1,1))
cat("\nmaximum values observed in the background:\n")
print(round(sapply(r[,-c(1:4)], max, na.rm=TRUE), 4))

**Note the maximum observed values** for both the pairwise Fst and the PBS. You will
compare the next region against them.

### A not-so-randomly chosen region on chromosome 5

Now do exactly the same for a 1 Mb region on chromosome 5. Note the extra filters:
`-baq 1 -C 50` adjust base qualities around indels, and `-minInd 8` requires data for at
least 8 of the 10 individuals.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh
# ~2-3 min

# the same three populations, for the region on chr 5
find $BAMFOLDERchr5 | grep bam$ | grep YRI > YRIchr5.filelist
find $BAMFOLDERchr5 | grep bam$ | grep JPT > JPTchr5.filelist
find $BAMFOLDERchr5 | grep bam$ | grep CEU > CEUchr5.filelist

FILTERS5="-minMapQ 30 -minQ 20 -baq 1 -C 50 -minInd 8"

# site frequency likelihoods
$ANGSD -b YRIchr5.filelist -anc $ANC -out yriChr5 $FILTERS5 $OPT -ref $REF
$ANGSD -b JPTchr5.filelist -anc $ANC -out jptChr5 $FILTERS5 $OPT -ref $REF
$ANGSD -b CEUchr5.filelist -anc $ANC -out ceuChr5 $FILTERS5 $OPT -ref $REF

# the 2D spectra
$REAL yriChr5.saf.idx ceuChr5.saf.idx > yri.ceuChr5.ml
$REAL yriChr5.saf.idx jptChr5.saf.idx > yri.jptChr5.ml
$REAL jptChr5.saf.idx ceuChr5.saf.idx > jpt.ceuChr5.ml

# Fst and PBS in sliding windows
$REAL fst index yriChr5.saf.idx jptChr5.saf.idx ceuChr5.saf.idx -fstout yri.jpt.ceuChr5 \
    -sfs yri.jptChr5.ml -sfs yri.ceuChr5.ml -sfs jpt.ceuChr5.ml
$REAL fst stats2 yri.jpt.ceuChr5.fst.idx -win 50000 -step 10000 > slidingwindowChr5

echo "--- windows: $(wc -l < slidingwindowChr5) ---"
head -n 3 slidingwindowChr5

Plot Fst and PBS along the region.

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

r <- read.delim("slidingwindowChr5", as.is=T, head=T)
names(r)[-c(1:4)] <- c("wFst_YRI_JPT","wFst_YRI_CEU","wFst_JPT_CEU",
                       "PBS_YRI","PBS_JPT","PBS_CEU")

par(mfrow=1:2)

plot(r$midPos, r$wFst_YRI_CEU, ylim=c(0,max(r$wFst_YRI_CEU)), type="b", pch=18,
     ylab="Fst", xlab="position on Chr 5")
points(r$midPos, r$wFst_YRI_JPT, col=2, type="b", pch=18)
points(r$midPos, r$wFst_JPT_CEU, col=3, type="b", pch=18)
legend("topleft", fill=1:3, c("YRI vs. CEU","YRI vs. JPT","JPT vs CEU"))

plot(r$midPos, r$PBS_YRI, ylim=c(0,max(r$PBS_CEU)), type="b", pch=18,
     ylab="PBS", xlab="position on Chr 5")
points(r$midPos, r$PBS_JPT, col=2, type="b", pch=18)
points(r$midPos, r$PBS_CEU, col=3, type="b", pch=18)
legend("topleft", fill=1:3, c("YRI","JPT","CEU"))

par(mfrow=c(1,1))
cat("\nmaximum values in this region:\n")
print(round(sapply(r[,-c(1:4)], max, na.rm=TRUE), 4))

**Questions**

 - Compare the values here with the background distribution you made above - the
   histograms and the maxima. Is this region extreme?
 - Why are there **two** peaks for the Fst and only **one** for the PBS?
 - In which of the three populations is this locus under selection?

Now find out which gene this is. Go to the [UCSC browser](https://genome-euro.ucsc.edu/cgi-bin/hgGateway),
choose Genome Browser, choose human **GRCh37/hg19**, and find the region. Read about the
gene on Wikipedia and see whether it fits the PBS result.

## 4. Bonus: what if we call genotypes instead?

Everything above avoided calling genotypes. Let us see what happens if we call SNPs and
genotypes first, the way GATK would, and build the spectrum from the calls. Skip this part
if you are running out of time.

Note that the output prefixes here are `yriGeno`, `jptGeno` and `ceuGeno`, so these runs
do not overwrite the saf results from the first section.

In [ ]:
source ~/sfs_fst_pbs_human/env.sh
# ~2-3 min

FILTERS2="-minMapQ 30 -minQ 20 -minInd 10"
OPT2="-gl 2 -doGeno 2 -doPost 2 -doMajorMinor 4 -doMaf 1 -SNP_pval 1e-6 -postCutoff 0.95"

$ANGSD -b YRI.filelist -out yriGeno $FILTERS2 $OPT2 -ref $REF -anc $ANC &
$ANGSD -b JPT.filelist -out jptGeno $FILTERS2 $OPT2 -ref $REF -anc $ANC &
$ANGSD -b CEU.filelist -out ceuGeno $FILTERS2 $OPT2 -ref $REF -anc $ANC
wait

echo "--- created files ---"
ls -l *Geno.geno.gz

While that runs, look at the options:

 - `-minInd 10` minimum number of individuals with data - here, with a called genotype.
   Why do we need this?
 - `-doGeno 2` print the counts (0,1,2) rather than the bases (AA, AT, TT)
 - `-doPost 2` use a uniform prior, i.e. call the genotype with the highest likelihood
 - `-doMajorMinor 4` use the ancestral allele from the chimp
 - `-doMaf 1 -SNP_pval 1e-6` the p-value cutoff for calling a SNP. What would happen to
   the SFS if you changed this threshold?
 - `-postCutoff 0.95` only call a genotype if its probability is above 0.95

In [ ]:
setwd(path.expand("~/sfs_fst_pbs_human"))

nnorm  <- function(x) x/sum(x)
getSFS <- function(x) table(factor(rowSums(read.table(x)[,-c(1:2)]), levels=1:20))

resG <- rbind(
  YRI = getSFS("yriGeno.geno.gz"),
  JPT = getSFS("jptGeno.geno.gz"),
  CEU = getSFS("ceuGeno.geno.gz")
)
colnames(resG) <- 1:20

# density instead of counts
resG <- t(apply(resG, 1, nnorm))

barplot(resG, beside=T, legend=c("YRI","JPT","CEU"), names=1:20,
        main="SFS from called genotypes")

In [ ]:
# the polymorphic sites only
resGPoly <- t(apply(resG[,-20], 1, nnorm))
barplot(resGPoly, beside=T, legend=c("YRI","JPT","CEU"), names=1:19,
        main="SFS from called genotypes, polymorphic sites")

In [ ]:
# downsampled to 5 individuals (10 chromosomes), fixed derived excluded
downsampleSFS <- function(x, chr){
    n <- length(x)
    mat <- sapply(1:chr, function(i) choose(1:n,i)*choose(n-(1:n),chr-i)/choose(n,chr))
    nnorm( as.vector(t(mat) %*% x)[-chr] )
}
resGDown <- t(apply(resG, 1, downsampleSFS, chr=10))
barplot(resGDown, beside=T, legend=c("YRI","JPT","CEU"), names=1:9,
        main="called genotypes, downsampled")

**Question**

 - How does this compare with the likelihood based estimates from section 2? Put the two
   downsampled barplots next to each other. There is also a rendered comparison
   [here](https://www.popgen.dk/albrecht/phdcourse/html/plots/realSFS.pdf).

Think about which direction the bias goes at 2-6X depth, and why.